# ch02 Final
This notebook provides a cleaner, more customizable version of the chapter 2 workflow:
- download and read text data
- preprocess tokens
- build a toy vocabulary and tokenizer
- encode / decode examples
- optional real tokenizer usage with `tiktoken`
- sliding-window dataset creation for training

In [1]:
import os
import re
import urllib.request
from pathlib import Path

DATA_PATH = Path("the-verdict.txt")
SOURCE_URL = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt"
)

if not DATA_PATH.exists():
    print(f"Downloading {DATA_PATH.name}...")
    urllib.request.urlretrieve(SOURCE_URL, DATA_PATH)
    print("Download complete.")

with DATA_PATH.open("r", encoding="utf-8") as f:
    raw_text = f.read()

print("Loaded text length:", len(raw_text))

Loaded text length: 20479


In [2]:
# Split text into tokens and punctuation, preserving punctuation as separate tokens.
TOKEN_PATTERN = r'([,.:;?_!"()\'\"]|--|\s)'
preprocessed = re.split(TOKEN_PATTERN, raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print("Sample tokens:", preprocessed[:30])
print("Total tokens after preprocessing:", len(preprocessed))

Sample tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
Total tokens after preprocessing: 4690


In [3]:
# Preserve first-seen order for a simple toy vocabulary.
unique_tokens = list(dict.fromkeys(preprocessed))
SPECIAL_TOKENS = ["<|endoftext|>", "<|unk|>"]

vocab_tokens = unique_tokens + [token for token in SPECIAL_TOKENS if token not in unique_tokens]

vocab = {token: idx for idx, token in enumerate(vocab_tokens)}
reverse_vocab = {idx: token for token, idx in vocab.items()}

print("Toy vocab size:", len(vocab))
print("First 20 vocab tokens:", vocab_tokens[:20])

Toy vocab size: 1132
First 20 vocab tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'good', 'fellow', 'enough', 'so', 'it', 'was', 'no', 'great']


## Toy tokenizer class
This tokenizer is exact-match based and uses a simple preprocessing rule.
Unknown tokens are mapped to `<|unk|>`.

In [4]:
class SimpleTokenizerV2:
    def __init__(self, vocab, unknown_token="<|unk|>"):
        self.str_to_int = vocab
        self.int_to_str = {idx: tok for tok, idx in vocab.items()}
        self.unknown_token = unknown_token

    def encode(self, text):
        tokens = re.split(TOKEN_PATTERN, text)
        tokens = [item.strip() for item in tokens if item.strip()]
        token_ids = [
            self.str_to_int[token] if token in self.str_to_int else self.str_to_int[self.unknown_token]
            for token in tokens
        ]
        return token_ids

    def decode(self, ids):
        tokens = [self.int_to_str.get(i, self.unknown_token) for i in ids]
        text = " ".join(tokens)
        text = re.sub(r"\s+([,.:;?_!\"()'])", r"\1", text)
        text = re.sub(r"\(\s+", "(", text)
        text = re.sub(r"\s+\)", ")", text)
        return text

    def token_to_id(self, token):
        return self.str_to_int.get(token, self.str_to_int[self.unknown_token])

    def id_to_token(self, idx):
        return self.int_to_str.get(idx, self.unknown_token)


simp = SimpleTokenizerV2(vocab)

In [5]:
text_example = "\"It's the last he painted, you know,\" Mrs. Gisburn said with pardonable pride."
encoded = simp.encode(text_example)
decoded = simp.decode(encoded)

print("Original text:", text_example)
print("Encoded ids:", encoded)
print("Decoded text:", decoded)

Original text: "It's the last he painted, you know," Mrs. Gisburn said with pardonable pride.
Encoded ids: [55, 200, 72, 73, 27, 64, 32, 668, 25, 477, 478, 25, 55, 61, 45, 5, 445, 124, 675, 676, 45]
Decoded text: " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


## Optional: `tiktoken` real GPT-2 tokenization
If `tiktoken` is available, this section shows how to use GPT-2 encoding/decoding.

In [6]:
try:
    import tiktoken

    tokenizer = tiktoken.get_encoding("gpt2")
    sample_text = "I am Simran Agarwal, Who are you?"
    sample_ids = tokenizer.encode(sample_text, allowed_special={"<|endoftext|>"})
    sample_decoded = tokenizer.decode(sample_ids)

    print("GPT-2 ids:", sample_ids)
    print("GPT-2 decoded:", sample_decoded)
except ImportError:
    print("tiktoken is not installed in this environment.")
    tokenizer = None

GPT-2 ids: [40, 716, 3184, 2596, 2449, 283, 16783, 11, 5338, 389, 345, 30]
GPT-2 decoded: I am Simran Agarwal, Who are you?


## Sliding window dataset for next-token training
This dataset tokenizes the entire text and creates overlapping sequences of length `max_length`.

In [7]:
try:
    import torch
    from torch.utils.data import Dataset, DataLoader

    class GPTDatasetV1(Dataset):
        def __init__(self, txt, tokenizer, max_length, stride):
            self.input_ids = []
            self.target_ids = []
            token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
            assert len(token_ids) > max_length, "The text must be longer than max_length."

            for i in range(0, len(token_ids) - max_length, stride):
                input_chunk = token_ids[i : i + max_length]
                target_chunk = token_ids[i + 1 : i + max_length + 1]
                self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
                self.target_ids.append(torch.tensor(target_chunk, dtype=torch.long))

        def __len__(self):
            return len(self.input_ids)

        def __getitem__(self, idx):
            return self.input_ids[idx], self.target_ids[idx]

    def create_dataloader_v1(txt, tokenizer, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
        dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            drop_last=drop_last,
            num_workers=num_workers,
        )

    if tokenizer is not None:
        max_length = 64
        dataloader = create_dataloader_v1(raw_text, tokenizer, batch_size=8, max_length=max_length, stride=max_length)
        inputs, targets = next(iter(dataloader))
        print("Input batch shape:", inputs.shape)
        print("Target batch shape:", targets.shape)

        vocab_size = 50257
        output_dim = 256
        token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
        pos_embedding_layer = torch.nn.Embedding(max_length, output_dim)

        token_embeddings = token_embedding_layer(inputs)
        pos_embeddings = pos_embedding_layer(torch.arange(max_length))
        input_embeddings = token_embeddings + pos_embeddings

        print("Token embeddings shape:", token_embeddings.shape)
        print("Positional embeddings shape:", pos_embeddings.shape)
        print("Combined embeddings shape:", input_embeddings.shape)
    else:
        print("Skipping dataset creation because tiktoken is unavailable.")
except ImportError as exc:
    print("PyTorch is not installed or could not be imported:", exc)

Input batch shape: torch.Size([8, 64])
Target batch shape: torch.Size([8, 64])
Token embeddings shape: torch.Size([8, 64, 256])
Positional embeddings shape: torch.Size([64, 256])
Combined embeddings shape: torch.Size([8, 64, 256])


### Customization points
- `TOKEN_PATTERN`: change tokenization behavior
- `SPECIAL_TOKENS`: adjust special tokens
- `max_length` and `stride`: control dataset chunking
- use `SimpleTokenizerV2` for toy experiments or `tiktoken` for real GPT-2 token ids